# ML_U2_Lab03 - Arboles de Decision## Laboratorio (1 hora) | Individual**Unidad:** 2 - Modelos de clasificacion supervisada**Modalidad:** Laboratorio computacional**Duracion estimada:** 60 minutos**Version:** 2025-1 | modificado: 2026-04-19---## Audiencias duales: Pregrado vs. DoctoradoEste notebook esta disenado para dos cursos simultaneamente:- **Pregrado:** Enfoque practico. Obligatorios: Partes 1, 2, 3 + preguntas azules.- **Doctorado:** Enfoque teorico-formal. Obligatorios: Todo lo anterior + secciones [PhD].

## Instrucciones| Aspecto | Pregrado | Doctorado ||---------|----------|-----------|| Obligatorio | Partes 1, 2, 3 | Todo + [PhD] || Opcional | Bonus azul | Bonus amarillo || Tiempo | 60 min | 90 min |

## Estructura| Parte | Tema | Tiempo ||-------|------|--------|| 1 | Sin computador: Gini | 10 min || 2 | Entrenamiento y visualizacion | 20 min || 3 | Efecto de profundidad | 20 min || Final | Analisis e interpretacion | 10 min || Bonus | Path tracing / Implementacion | Opcional |

---# SETUP - NO MODIFICAREjecuta esta celda para cargar librerias y datos.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import load_breast_cancerfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.tree import DecisionTreeClassifier, plot_treefrom sklearn.metrics import accuracy_score, f1_score, confusion_matriximport warningswarnings.filterwarnings('ignore')RANDOM_STATE = 42np.random.seed(RANDOM_STATE)data = load_breast_cancer()X = pd.DataFrame(data.data, columns=data.feature_names)y = pd.Series(data.target, name='target')print("✅ Setup completo")print(f"   Dataset: breast_cancer (n={X.shape[0]}, features={X.shape[1]})")print(f"   Classes: {np.unique(y)} (0=Maligno, 1=Benigno)")print(f"   Distribution: {np.bincount(y)}")

---# PARTE 1 - Sin computador: Calculo manual de Gini (10 min)## ObjetivoEntender como mide el arbol de decision la pureza usando Gini.## Problema: Prediccion de lluvia| ID | Temperatura | Humedad | Lluvia ||----|-------------|---------|--------|| 1  | 20          | 65      | 0      || 2  | 22          | 68      | 0      || 3  | 28          | 75      | 1      || 4  | 30          | 80      | 1      |## Tarea 1: Calcular Gini del nodo raizGini(S) = 1 - sum(p_c^2)Tenemos 2 muestras clase 0 y 2 muestras clase 1.p0 = 0.5, p1 = 0.5Gini(raiz) = 1 - (0.25 + 0.25) = 0.5## Tarea 2: Ganancia con division Temperatura > 25Rama izquierda (temp <= 25): [0, 0] -> Gini = 0.0Rama derecha (temp > 25): [1, 1] -> Gini = 0.0Gini ponderado = 0.5*0 + 0.5*0 = 0.0Ganancia = 0.5 - 0.0 = 0.5

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1; padding:14px 18px; border-radius:6px;"><b>PREGRADO:</b> Obtuviste 0.5 y 0.5? Perfecto\!</div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D; padding:14px 18px; border-radius:6px;"><b>DOCTORADO:</b> Por que Gini vs Entropia? H(S)=1 bit representa?</div>

In [ ]:
y_raiz = np.array([0, 0, 1, 1])y_izq = np.array([0, 0])y_der = np.array([1, 1])def gini(y):    _, counts = np.unique(y, return_counts=True)    p = counts / len(y)    return 1 - np.sum(p**2)gini_raiz = gini(y_raiz)gini_izq = gini(y_izq)gini_der = gini(y_der)gini_pond = (2/4) * gini_izq + (2/4) * gini_dergain = gini_raiz - gini_pondprint("="*60)print("VERIFICACION DE CALCULOS")print("="*60)print(f"Gini(raiz) = {gini_raiz:.4f} {'OK' if abs(gini_raiz-0.5)<0.001 else 'ERROR'}")print(f"Gini(izq)  = {gini_izq:.4f} {'OK' if abs(gini_izq-0.0)<0.001 else 'ERROR'}")print(f"Gini(der)  = {gini_der:.4f} {'OK' if abs(gini_der-0.0)<0.001 else 'ERROR'}")print(f"Ganancia = {gain:.4f} {'OK' if abs(gain-0.5)<0.001 else 'ERROR'}')

---# PARTE 2 - Entrenamiento y visualizacion del arbol (20 min)

<b>TODO 1:</b> Realiza un split estratificado 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")print(f"y_train: {np.bincount(y_train)}, y_test: {np.bincount(y_test)}")

In [ ]:
try:    assert X_train.shape[0] + X_test.shape[0] == X.shape[0]    assert 0.78 <= X_train.shape[0]/X.shape[0] <= 0.82    print("OK - Split correcto")except AssertionError:    print("ERROR")

<b>TODO 2:</b> Entrena DecisionTreeClassifier con max_depth=3

In [ ]:
tree_model = DecisionTreeClassifier(    max_depth=3, criterion='gini', random_state=RANDOM_STATE)tree_model.fit(X_train, y_train)print(f"Modelo entrenado: {tree_model.tree_.node_count} nodos, profundidad {tree_model.get_depth()}")

<b>TODO 3:</b> Visualiza el arbol con figsize=(20,12)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 12))plot_tree(    tree_model,    feature_names=X.columns,    class_names=['Maligno', 'Benigno'],    filled=True,    rounded=True,    ax=ax)plt.title("Arbol de Decision (max_depth=3)", fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("Arbol visualizado")

<b>TODO 4:</b> Calcula accuracy y F1-score en test

In [ ]:
y_pred = tree_model.predict(X_test)test_acc = accuracy_score(y_test, y_pred)test_f1 = f1_score(y_test, y_pred)print("="*60)print("RENDIMIENTO DEL MODELO")print("="*60)print(f"Accuracy: {test_acc:.4f}")print(f"F1-score: {test_f1:.4f}")cm = confusion_matrix(y_test, y_pred)print(f"Matriz: {cm}")

In [ ]:
try:    assert hasattr(tree_model, 'tree_')    assert 0 <= test_acc <= 1    assert 0 <= test_f1 <= 1    print("OK - Modelo y metricas validas")except:    print("ERROR")

---# PARTE 3 - Efecto de la profundidad (20 min)

<b>TODO 5:</b> Barre max_depth de 1 a 20

In [ ]:
max_depths = list(range(1, 21))train_scores = []cv_scores = []cv_stds = []for depth in max_depths:    dt = DecisionTreeClassifier(        max_depth=depth, criterion='gini', random_state=RANDOM_STATE    )    dt.fit(X_train, y_train)    train_scores.append(dt.score(X_train, y_train))        cv_fold = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')    cv_scores.append(cv_fold.mean())    cv_stds.append(cv_fold.std())train_scores = np.array(train_scores)cv_scores = np.array(cv_scores)cv_stds = np.array(cv_stds)print("Primeros 5 valores:")for i in range(5):    print(f"  depth={max_depths[i]}: train={train_scores[i]:.4f}, cv={cv_scores[i]:.4f}")

In [ ]:
try:    assert len(train_scores) == 20    assert len(cv_scores) == 20    assert np.all((train_scores >= 0) & (train_scores <= 1))    print("OK - Arrays correctos")except:    print("ERROR")

<b>TODO 6:</b> Grafica la curva de aprendizaje

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))ax.plot(max_depths, train_scores, 'o-', label='Train', linewidth=2)ax.plot(max_depths, cv_scores, 's-', label='CV', linewidth=2)ax.set_xlabel('max_depth', fontsize=12, fontweight='bold')ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')ax.set_title('Curva de Aprendizaje', fontsize=13, fontweight='bold')ax.grid(True, alpha=0.3)ax.legend()ax.set_xticks(max_depths)ax.set_ylim([0.85, 1.02])plt.tight_layout()plt.show()

<b>TODO 7 [PhD]:</b> Agrega barras de error (CV ± 1 std)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))ax.plot(max_depths, train_scores, 'o-', label='Train', linewidth=2)ax.plot(max_depths, cv_scores, 's-', label='CV', linewidth=2)ax.fill_between(    max_depths,    cv_scores - cv_stds,    cv_scores + cv_stds,    alpha=0.2,    label='CV +/- 1std')ax.set_xlabel('max_depth', fontsize=12, fontweight='bold')ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')ax.set_title('Curva de Aprendizaje con barras de error', fontsize=13, fontweight='bold')ax.grid(True, alpha=0.3)ax.legend()ax.set_xticks(max_depths)plt.tight_layout()plt.show()

<b>TODO 8:</b> Selecciona el mejor max_depth

In [ ]:
best_idx = np.argmax(cv_scores)best_depth = max_depths[best_idx]print("="*60)print("SELECCION DEL MEJOR max_depth")print("="*60)print(f"Mejor max_depth: {best_depth}")print(f"  CV Accuracy: {cv_scores[best_idx]:.4f}")print(f"  Train Accuracy: {train_scores[best_idx]:.4f}")print(f"  Diferencia: {train_scores[best_idx] - cv_scores[best_idx]:.4f}")best_model = DecisionTreeClassifier(    max_depth=best_depth,    criterion='gini',    random_state=RANDOM_STATE)best_model.fit(X_train, y_train)y_pred_best = best_model.predict(X_test)best_acc = accuracy_score(y_test, y_pred_best)best_f1 = f1_score(y_test, y_pred_best)print(f"\nRendimiento en TEST:")print(f"  Accuracy: {best_acc:.4f}")print(f"  F1: {best_f1:.4f}")

In [ ]:
try:    assert 1 <= best_depth <= 20    assert hasattr(best_model, 'tree_')    assert 0 <= best_acc <= 1    print("OK - Modelo final valido")except:    print("ERROR")

---# PARTE FINAL - Analisis e Interpretacion (10 min)

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1; padding:14px 18px; border-radius:6px;"><b>PREGRADO: Preguntas de interpretacion</b><br><br>1. Feature mas importante? Sentido medico?<br><br>2. Para minimizar falsos negativos, que cambiarías?<br><br>3. Explica el arbol a un medico no tecnico</div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D; padding:14px 18px; border-radius:6px;"><b>DOCTORADO: Preguntas criticas</b><br><br>1. MDI sesgada? Alternativas (permutation, SHAP)?<br><br>2. Arbol greedy. Existe otro mejor de igual profundidad?<br><br>3. Experimento para estimar error de Bayes</div>

## Feature Importance

In [ ]:
fi = pd.DataFrame({    'feature': X.columns,    'importance': best_model.feature_importances_}).sort_values('importance', ascending=False)print("="*60)print("FEATURE IMPORTANCE")print("="*60)print(fi.head(10).to_string(index=False))fig, ax = plt.subplots(figsize=(10, 8))top = fi.head(15)ax.barh(range(len(top)), top['importance'], color='#2E86C1')ax.set_yticks(range(len(top)))ax.set_yticklabels(top['feature'])ax.set_xlabel('Importance (MDI)', fontsize=12, fontweight='bold')ax.set_title('Top 15 Features', fontsize=13, fontweight='bold')ax.invert_yaxis()plt.tight_layout()plt.show()

---# BONUS

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1; padding:14px 18px; border-radius:6px;"><b>PREGRADO: Path Tracing</b><br><br>Explica una prediccion individual trazando el camino</div>

In [ ]:
sidx = 0xs = X_test.iloc[sidx:sidx+1]ys = y_test.iloc[sidx]pred = best_model.predict(xs)[0]prob = best_model.predict_proba(xs)[0]print(f"Muestra {sidx}:")print(xs.to_string())print(f"Real: {ys}, Prediccion: {pred}")print(f"Probabilidades: {prob}")print(f"Confianza: {max(prob)*100:.2f}%")path = best_model.decision_path(xs)tree = best_model.tree_print(f"\nCamino desde raiz:")for nid in path.indices[0][:-1]:    fid = tree.feature[nid]    fname = X.columns[fid]    fval = xs[fname].values[0]    thresh = tree.threshold[nid]    d = "izquierda" if fval <= thresh else "derecha"    print(f"  {fname}={fval:.2f} vs {thresh:.2f} -> {d}")

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D; padding:14px 18px; border-radius:6px;"><b>DOCTORADO: Implementacion desde cero</b></div>

In [ ]:
def gini_index(y):    _, counts = np.unique(y, return_counts=True)    p = counts / len(y)    return 1 - np.sum(p**2)def find_best_split(X, y, feature_idx):    fvals = X.iloc[:, feature_idx].values    unique_vals = np.unique(fvals)[:-1]        best_gain = -np.inf    best_thresh = None        for thresh in unique_vals:        left_mask = fvals <= thresh        right_mask = ~left_mask                y_left = y[left_mask]        y_right = y[right_mask]                if len(y_left) == 0 or len(y_right) == 0:            continue                gini_parent = gini_index(y)        gini_left = gini_index(y_left)        gini_right = gini_index(y_right)                gini_weighted = (len(y_left)/len(y)) * gini_left + (len(y_right)/len(y)) * gini_right        gain = gini_parent - gini_weighted                if gain > best_gain:            best_gain = gain            best_thresh = thresh        return best_thresh, best_gainfidx = 0fname = X.columns[fidx]thresh, gain = find_best_split(X_train, y_train, fidx)print(f"Feature: {fname}")print(f"Mejor umbral: {thresh:.4f}")print(f"Ganancia: {gain:.4f}")dt_sk = DecisionTreeClassifier(max_depth=1, criterion='gini', random_state=RANDOM_STATE)dt_sk.fit(X_train, y_train)sklearn_thresh = dt_sk.tree_.threshold[0]print(f"sklearn umbral: {sklearn_thresh:.4f}")print(f"Match: {'SI' if abs(thresh - sklearn_thresh) < 0.0001 else 'NO'}')

---# CHECKLIST DE ENTREGA## Pregrado- [ ] Parte 1: Gini=0.5, Gain=0.5- [ ] TODO 1: Split 80/20- [ ] TODO 2: Modelo max_depth=3- [ ] TODO 3: Visualizacion (20x12)- [ ] TODO 4: Accuracy y F1- [ ] TODO 5: Barrida 1-20- [ ] TODO 6: Curva aprendizaje- [ ] TODO 8: Mejor depth- [ ] Preguntas azules respondidas- [ ] Bonus (opcional)## Doctorado (todo anterior +)- [ ] TODO 7: Barras de error- [ ] Preguntas amarillas respondidas- [ ] Bonus (opcional)## Final- [ ] Notebook ejecuta sin errores- [ ] Tests de sanidad pasan- [ ] Figuras legibles- [ ] Archivo: ML_U2_Lab03_arboles_decision.ipynbFecha: 2026-04-19 | Version: 2025-1